# Model 0

Develop a model specification that includes alternative specific constant, cost and travel time for each alternative.
Cost and travel time are associated with generic parameters.
Present both the specification (i.e., the utility functions) and the estimation results (parameter values, t-tests or p-values, null and final log-likelihoods). 

# Importing packages

We first import some useful Python packages. pandas is an open source data analysis and manipulation tool built in the Python programming language. Biogeme is a Python package for estimating discrete choice models to explain and predict how people choose among alternatives.

In [ ]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from tqdm.notebook import tqdm
from biogeme.expressions import Beta, Variable, log, exp
from biogeme.results_processing import get_pandas_estimated_parameters, html_output

# Reading and preparing the data

We first import the data into Pandas. We read the data from a text file, where the data are separated by tabs. 

In [ ]:
df = pd.read_csv('lpmc19.dat', sep='\t')
df

We then import this database into Biogeme.

In [ ]:
database = db.Database('netherlands', df)

We identify the columns that will be used as variables in our model. 

In [ ]:
trip_id = Variable('trip_id')
travel_mode = Variable('travel_mode')
purpose = Variable('purpose')
distance = Variable('distance')
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
dur_pt_access = Variable('dur_pt_access')
dur_pt_rail  = Variable('dur_pt_rail ')
dur_pt_bus  = Variable('dur_pt_bus')
dur_pt_int = Variable('dur_pt_int')
dur_driving = Variable('dur_driving')
cost_transit = Variable('cost_transit')
cost_driving_fuel = Variable('cost_driving_fuel ')
cost_driving_ccharge = Variable('cost_driving_ccharge')

In [ ]:
database.num_rows()

We define the total travel time by public transportation

In [ ]:
dur_pt_total = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int
cost_car = cost_driving_fuel + cost_driving_ccharge

We convert the Guilders into Euros.

In [ ]:
POUNDS_TO_EUROS = 1.16
cost_car_euro = cost_car * POUNDS_TO_EUROS
cost_pt_euro = cost_transit * POUNDS_TO_EUROS

# Model specification

We can now specify a model that contains 4 ASC

In [ ]:
asc_car = Beta('asc_car', 0, None, None, 0)
asc_walk = Beta('asc_walk', 0, None, None, 0)
asc_bike = Beta('asc_bike', 0, None, None, 0)
asc_pt = Beta('asc_pt', 0, None, None, 0)
beta_cost_car = Beta('beta_cost_car', 0, None, None, 0)
beta_time_car = Beta('beta_time_car', 0, None, None, 0)
beta_cost_pt = Beta('beta_cost_pt', 0, None, None, 0)
beta_time_pt = Beta('beta_time_pt', 0, None, None, 0)
beta_time_bike = Beta('beta_time_bike', 0, None, None, 0)

We  write the utility functions:
**We choose walking as the reference**

In [ ]:
v_car = asc_car + beta_cost_car * cost_car + beta_time_car * dur_driving
v_walk = 0
v_bike = asc_bike + beta_time_bike * dur_cycling
v_pt = asc_pt + beta_cost_pt * cost_transit + beta_time_pt * dur_pt_total

And we write the choice model:

In [ ]:
prob_car = 1 / (1 + exp(v_rail - v_car))
prob_rail = 1 - prob_car

The formula of the contribution of each observation to the log likelihood function depends on the observed choice: 

In [ ]:
prob_observation = prob_car * (choice == 0) + prob_rail * (choice == 1)
logprob = log(prob_observation)

We initialize Biogeme with this expression and the database. We call the model 'binary_netherlands'.

In [ ]:
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'binary_netherlands'

# Model estimation

Biogeme tries to read a file `__binary_netherlands.iter` containing intermediary results from a previous estimation run. If it does not find it, it triggers a warning that can be ignored.

In [ ]:
results = biogeme.estimate()

The estimation results are stored in a object. The help function shows a a detailed description. 

In [ ]:
help(results)

We print some general statistics: 

In [ ]:
print(results.print_general_statistics())

We print the estimation results

In [ ]:
get_pandas_estimated_parameters(estimation_results=results)

The results are also available in an HTML file than can be opened in a browser.

In [ ]:
html_output.generate_html_file(results, 'binary_netherlands')

# Exercise

You want to improve the model specification by including the attributes travel cost and travel time. Try the following model specifications:
1. the coefficients of both attributes are generic.
2. the coefficients of both attributes are alternative specific.
3. the time coefficient is generic, and the cost coefficient is alternative-specific.
4. the cost coefficient is generic, and the time coefficient is alternative-specific.

Comment on the results and explain which model you prefer.

1. Model 1 : Coefficients of both attributes are generic

In [ ]:
asc_car1 = Beta('asc_car1', 0, None, None, 0)
beta_cost = Beta('beta_cost', 0, None, None, 0)
beta_time = Beta('beta_time', 0, None, None, 0)

In [ ]:
v_car1 = asc_car1 + beta_cost * car_cost + beta_time * car_time
v_rail1 = beta_cost * rail_cost + beta_time * rail_time

In [ ]:
prob_car1 = 1 / (1 + exp(v_rail1 - v_car1))
prob_rail1 = 1 - prob_car1

In [ ]:
prob_observation1 = prob_car1 * (choice == 0) + prob_rail1 * (choice == 1)
logprob1 = log(prob_observation1)

In [ ]:
biogeme1 = bio.BIOGEME(database, logprob1)
biogeme1.model_name = 'binary_netherlands_model_1'

In [ ]:
results1 = biogeme1.estimate()
print(results1.print_general_statistics())
get_pandas_estimated_parameters(estimation_results=results1)

2. Model 2 : Coefficient of both attributes are alternative specific

In [ ]:
asc_car2 = Beta('asc_car2', 0, None, None, 0)
beta_car_cost = Beta('beta_car_cost', 0, None, None, 0)
beta_rail_cost = Beta('beta_rail_cost', 0, None, None, 0)
beta_car_time = Beta('beta_car_time', 0, None, None, 0)
beta_rail_time = Beta('beta_rail_time', 0, None, None, 0)

v_car2 = asc_car2 + beta_car_cost * car_cost + beta_car_time * car_time
v_rail2 = beta_rail_cost * rail_cost + beta_rail_time * rail_time

prob_car2 = 1 / (1 + exp(v_rail2 - v_car2))
prob_rail2 = 1 - prob_car2

prob_observation2 = prob_car2 * (choice == 0) + prob_rail2 * (choice == 1)
logprob2 = log(prob_observation2)

biogeme2 = bio.BIOGEME(database, logprob2)
biogeme2.model_name = 'binary_netherlands_model_2'

In [ ]:
results2 = biogeme2.estimate()
print(results2.print_general_statistics())
get_pandas_estimated_parameters(estimation_results=results2)

3. Model 3 : Time coefficient is generic, and the cost coefficient is alternative-specific

In [ ]:
asc_car3 = Beta('asc_car2', 0, None, None, 0)
beta_car_cost = Beta('beta_car_cost', 0, None, None, 0)
beta_rail_cost = Beta('beta_rail_cost', 0, None, None, 0)
beta_time = Beta('beta_time', 0, None, None, 0)

v_car3 = asc_car3 + beta_car_cost * car_cost + beta_time * car_time
v_rail3 = beta_rail_cost * rail_cost + beta_time * rail_time

prob_car3 = 1 / (1 + exp(v_rail3 - v_car3))
prob_rail3 = 1 - prob_car3

prob_observation3 = prob_car3 * (choice == 0) + prob_rail3 * (choice == 1)
logprob3 = log(prob_observation3)

biogeme3 = bio.BIOGEME(database, logprob3)
biogeme3.model_name = 'binary_netherlands_model_3'

In [ ]:
results3 = biogeme3.estimate()
print(results3.print_general_statistics())
get_pandas_estimated_parameters(estimation_results=results3)

4. Model 4 : Cost coefficient is generic, and the time coefficient is alternative-specific

In [ ]:
asc_car4 = Beta('asc_car4', 0, None, None, 0)
beta_cost = Beta('beta_cost', 0, None, None, 0)
beta_car_time = Beta('beta_car_time', 0, None, None, 0)
beta_rail_time = Beta('beta_rail_time', 0, None, None, 0)

v_car4 = asc_car4 + beta_cost * car_cost + beta_car_time * car_time
v_rail4 = beta_cost * rail_cost + beta_car_time * rail_time

prob_car4 = 1 / (1 + exp(v_rail4 - v_car4))
prob_rail4 = 1 - prob_car4

prob_observation4 = prob_car4 * (choice == 0) + prob_rail4 * (choice == 1)
logprob4 = log(prob_observation4)

biogeme4 = bio.BIOGEME(database, logprob4)
biogeme4.model_name = 'binary_netherlands_model_4'

In [ ]:
results4 = biogeme4.estimate()
print(results4.print_general_statistics())
get_pandas_estimated_parameters(estimation_results=results4)

**Response**

In every models, time seems to have a bigger impact than cost.

I would choose Model 2 because it got the highest Likelihood ratio, even though its rho square was a little higher than the others. I prefer be able to describe more observations while loosing a little of likelihood.